# Generador de Gráficos 1.1 a 1.3 (Estandarizado Villa Gaucho)

Este cuaderno implementa el generador estandarizado para las preguntas 1.1, 1.2 y 1.3 con las correcciones y especificaciones solicitadas:
- **Gráficos de Barras VERTICALES (Cuantitativas discretas)**:
  - Para 1.1.1, 1.2.1 y 1.3.1 (cantidad de personas).
  - **Regla del 2%**: Si el porcentaje es menor a 2%, la frecuencia absoluta aparece **afuera de la barra** arriba (`"X (Y,Y%)"`). Si es >= 2%, la frecuencia aparece **adentro de la barra** en blanco y negrita (tamaño 14).
  - Color institucional: **`#1b7882`**.
- **Gráficos de Torta (Cualitativas)**:
  - Para 1.1.2, 1.2.2 y 1.3.2 (género).
  - **Línea directa sin subrayar**: conecta la cuña directamente al inicio del texto.
  - Sin líneas blancas divisorias (`edgecolor='none'`).
  - Frecuencia absoluta adentro de cada cuña (tamaño 14).
  - Nombre de la categoría arriba y porcentaje abajo (`X,X%`).
  - Orden estricto de mayor a menor porcentaje con la paleta de Villa Gaucho: 1. Azul (`#4285F4`), 2. Naranja (`#FB8C00`), 3. Verde (`#34A853`), 4. Amarillo (`#FBBC05`), 5. Violeta (`#8E24AA`), 6. Beige (`#D7CCC8`).
  - Distribución balanceada verticalmente en el recuadro para evitar superposiciones.
- **Datos reales y verificados de 1.3.2**: toma correctamente `df_1_3` (Mujer: 20, Se desconoce: 19, Mujer - Varón: 7, Varón: 6).


In [ ]:
# ==============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN GLOBAL
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración global de tamaño de letra 14 para todo el cuaderno
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 14
plt.rcParams['figure.titlesize'] = 14

# Paleta para tortas (de mayor a menor porcentaje)
PALETA_TORTA = [
    '#4285F4',  # 1. Azul
    '#FB8C00',  # 2. Naranja
    '#34A853',  # 3. Verde
    '#FBBC05',  # 4. Amarillo
    '#8E24AA',  # 5. Violeta
    '#D7CCC8'   # 6. Beige
]

# Color estándar para barras de un solo color
COLOR_BARRAS_UNICO = '#1b7882'


In [ ]:
# ==============================================================================
# 2. FUNCIONES DE GRAFICACIÓN ESTANDARIZADAS
# ==============================================================================
def graficar_barras_verticales(etiquetas, valores, titulo="", xlabel="Cantidad de personas", ylabel="Cantidad de respuestas", figsize=(9, 6)):
    """
    Genera un gráfico de barras VERTICAL estandarizado:
    - Si porcentaje < 2%, frecuencia y porcentaje van afuera de la barra (arriba).
    - Si porcentaje >= 2%, frecuencia adentro de la barra (blanco, negrita) y porcentaje arriba.
    - Color #1b7882.
    """
    valores = np.array(valores, dtype=float)
    total = np.sum(valores)
    porcentajes = (valores / total) * 100 if total > 0 else np.zeros_like(valores)
    n = len(etiquetas)
    x_pos = np.arange(n)

    fig, ax = plt.subplots(figsize=figsize)

    barras = ax.bar(
        x_pos,
        valores,
        color=COLOR_BARRAS_UNICO,
        width=0.62,
        edgecolor='none'
    )

    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(lbl) for lbl in etiquetas], fontsize=14)

    max_h = max(valores) if len(valores) > 0 else 1

    for barra, cantidad, porcentaje in zip(barras, valores, porcentajes):
        h = barra.get_height()
        x = barra.get_x() + barra.get_width() / 2.0

        # REGLA DEL 2%:
        if porcentaje < 2.0:
            # Afuera de la barra (arriba)
            ax.text(
                x, h + (max_h * 0.02),
                f"{int(cantidad)}\n({porcentaje:.1f}%)".replace('.', ','),
                ha='center', va='bottom',
                color='#202124', fontsize=14, fontweight='medium'
            )
        else:
            # Frecuencia adentro y porcentaje arriba
            ax.text(
                x, h / 2.0, str(int(cantidad)),
                ha='center', va='center',
                color='white', fontweight='bold', fontsize=14
            )
            ax.text(
                x, h + (max_h * 0.02),
                f"{porcentaje:.1f}%".replace('.', ','),
                ha='center', va='bottom',
                color='#202124', fontsize=14
            )

    if titulo:
        ax.set_title(titulo, fontsize=14, pad=18, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=14, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=14, labelpad=10)

    ax.set_ylim(0, max_h * 1.25)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.show()


def graficar_torta(categorias, valores, titulo="", figsize=(11, 7), distancia_etiquetas_y=0.45):
    """
    Genera un gráfico de torta con el formato exacto del Informe Villa Gaucho:
    - Sin bordes blancos entre porciones.
    - Frecuencia absoluta adentro de cada cuña (tamaño 14).
    - Línea conector que arranca justo donde comienza el texto (sin subrayar).
    - Nombre de la categoría arriba, porcentaje abajo.
    - Ordenado de mayor a menor con la paleta institucional.
    - Distribución balanceada verticalmente en el recuadro.
    """
    df_temp = pd.DataFrame({'categoria': [str(c) for c in categorias], 'fi': list(valores)})
    df_temp = df_temp.sort_values(by='fi', ascending=False).reset_index(drop=True)
    colores = [PALETA_TORTA[i % len(PALETA_TORTA)] for i in range(len(df_temp))]

    cats = df_temp['categoria'].tolist()
    vals = df_temp['fi'].values
    total = sum(vals)
    pcts = (vals / total) * 100 if total > 0 else np.zeros_like(vals)

    fig, ax = plt.subplots(figsize=figsize, subplot_kw=dict(aspect="equal"))

    wedges, texts, autotexts = ax.pie(
        vals,
        autopct='%1.0f',
        pctdistance=0.68,
        colors=colores,
        startangle=90,
        counterclock=False,
        wedgeprops=dict(width=1, edgecolor='none')
    )

    for at, val in zip(autotexts, vals):
        at.set_text(str(int(val)))
        at.set_color('#202124')
        at.set_fontsize(14)
        at.set_fontweight('medium')

    wedge_items = []
    for i, p in enumerate(wedges):
        ang = (p.theta2 - p.theta1) / 2.0 + p.theta1
        rad = np.deg2rad(ang)
        x = np.cos(rad)
        y = np.sin(rad)
        lado = 1 if x >= 0 else -1
        wedge_items.append({
            'index': i,
            'ang': ang,
            'rad': rad,
            'x': x,
            'y': y,
            'lado': lado,
            'y_afuera': 1.15 * y
        })

    ax.set_xlim(-2.4, 2.4)
    ax.set_ylim(-1.6, 1.6)

    # Distribución balanceada vertical
    for lado in [1, -1]:
        group = [item for item in wedge_items if item['lado'] == lado]
        if not group:
            continue
        group.sort(key=lambda item: item['y'])
        k = len(group)
        if k == 1:
            group[0]['y_afuera'] = group[0]['y'] * 1.15
        else:
            span = (k - 1) * distancia_etiquetas_y
            avg_y = np.mean([item['y'] for item in group])
            start_y = avg_y - (span / 2.0)
            start_y = max(-1.30, min(1.30 - span, start_y))
            for idx_g, item in enumerate(group):
                item['y_afuera'] = start_y + idx_g * distancia_etiquetas_y

    for item in wedge_items:
        i = item['index']
        rad = item['rad']
        x_punto = 0.90 * np.cos(rad)
        y_punto = 0.90 * np.sin(rad)

        lado = item['lado']
        y_afuera = item['y_afuera']
        ha = 'left' if lado == 1 else 'right'
        pct_str = f"{pcts[i]:.1f}%".replace('.', ',')

        x_texto = 1.95 * lado

        # 1. Nombre de la categoría ARRIBA de la línea
        ax.text(
            x_texto, y_afuera + 0.03,
            cats[i],
            horizontalalignment=ha,
            verticalalignment='bottom',
            fontsize=14,
            color='#202124',
            fontweight='medium'
        )

        # 2. Porcentaje ABAJO de la línea
        ax.text(
            x_texto, y_afuera - 0.03,
            pct_str,
            horizontalalignment=ha,
            verticalalignment='top',
            fontsize=14,
            color='#757575',
            fontweight='normal'
        )

        # 3. Línea que conecta la porción con donde arranca el texto (SIN SUBRAYAR)
        ax.plot([x_punto, x_texto], [y_punto, y_afuera], color='#9e9e9e', lw=1.2)
        ax.plot(x_punto, y_punto, marker='o', markersize=4, color='#9e9e9e')

    if titulo:
        ax.set_title(titulo, fontsize=14, pad=25, fontweight='bold', color='#202124')

    plt.subplots_adjust(left=0.08, right=0.92, top=0.88, bottom=0.10)
    plt.show()


## 3. Carga y Procesamiento de Datos


In [ ]:
# Cargar el archivo CSV con los datos del proyecto
df = pd.read_csv('datos.csv')

# Subconjuntos de datos para las preguntas 1.1, 1.2 y 1.3
df_1_1 = df[['1.1.1 Cantidad de personas que definen qué se cocina en el hogar','1.1.2 Género de quién/es definen qué se cocina']]
df_1_2 = df[['1.2.1 Cantidad de personas que se encargan mayoritariamente de las compras','1.2.2 Género de quién/es se encargan de las compras']]
df_1_3 = df[['1.3.1 Cantidad de personas que se encargan de preparar las comidas en el hogar','1.3.2 Género de quién se encarga de preparar las comidas en el hogar']]

# Tablas de frecuencia para 1.1
conteo_1_1_cant = df_1_1['1.1.1 Cantidad de personas que definen qué se cocina en el hogar'].value_counts().sort_index()
conteo_1_1_gen = df_1_1['1.1.2 Género de quién/es definen qué se cocina'].value_counts()

# Tablas de frecuencia para 1.2
conteo_1_2_cant = df_1_2['1.2.1 Cantidad de personas que se encargan mayoritariamente de las compras'].value_counts().sort_index()
conteo_1_2_gen = df_1_2['1.2.2 Género de quién/es se encargan de las compras'].value_counts()

# Tablas de frecuencia para 1.3 (CORRECTAMENTE ASIGNADAS DESDE DF_1_3)
conteo_1_3_cant = df_1_3['1.3.1 Cantidad de personas que se encargan de preparar las comidas en el hogar'].value_counts().sort_index()
conteo_1_3_gen = df_1_3['1.3.2 Género de quién se encarga de preparar las comidas en el hogar'].value_counts()


## 4. Pregunta 1.1: Quiénes definen qué se cocina


In [ ]:
# Gráfico de barras VERTICAL: Cantidad de personas que definen qué se cocina
graficar_barras_verticales(
    etiquetas=conteo_1_1_cant.index,
    valores=conteo_1_1_cant.values,
    titulo="Cantidad de personas que definen qué se cocina en el hogar",
    xlabel="Cantidad de personas",
    ylabel="Cantidad de respuestas"
)


In [ ]:
# Gráfico de torta: Género de quién/es definen qué se cocina (Línea sin subrayar)
graficar_torta(
    categorias=conteo_1_1_gen.index,
    valores=conteo_1_1_gen.values,
    titulo="Género de quién/es definen qué se cocina en el hogar"
)


## 5. Pregunta 1.2: Quiénes se encargan de las compras


In [ ]:
# Gráfico de barras VERTICAL: Cantidad de personas que se encargan de las compras
graficar_barras_verticales(
    etiquetas=conteo_1_2_cant.index,
    valores=conteo_1_2_cant.values,
    titulo="Cantidad de personas que se encargan de las compras",
    xlabel="Cantidad de personas",
    ylabel="Cantidad de respuestas"
)


In [ ]:
# Gráfico de torta: Género de quién/es se encargan de las compras (Línea sin subrayar)
graficar_torta(
    categorias=conteo_1_2_gen.index,
    valores=conteo_1_2_gen.values,
    titulo="Género de quién/es se encargan de las compras"
)


## 6. Pregunta 1.3: Quiénes se encargan de preparar las comidas


In [ ]:
# Gráfico de barras VERTICAL: Cantidad de personas que preparan las comidas
graficar_barras_verticales(
    etiquetas=conteo_1_3_cant.index,
    valores=conteo_1_3_cant.values,
    titulo="Cantidad de personas que preparan las comidas en el hogar",
    xlabel="Cantidad de personas",
    ylabel="Cantidad de respuestas"
)


In [ ]:
# Gráfico de torta: Género de quién se encarga de preparar las comidas (Línea sin subrayar)
graficar_torta(
    categorias=conteo_1_3_gen.index,
    valores=conteo_1_3_gen.values,
    titulo="Género de quién/es preparan las comidas en el hogar"
)
